In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding, BatchNo
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.optimizers import Adam
import json
import pickle
import optuna
import os

ImportError: cannot import name 'BatchNo' from 'keras.layers' (/usr/local/lib/python3.12/dist-packages/keras/layers/__init__.py)

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

In [ ]:
RAND = 1337
np.random.seed(RAND)

In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "./"

data = pd.read_csv(os.path.join(DATA_DIR, './Data.csv'))
labels = pd.read_csv(os.path.join(DATA_DIR, './Label.csv'))
df = pd.concat([data, labels], axis=1)

In [ ]:
df.head()

,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,214392.0,9.0,21.0,388.0,24564.0,194.0,0.0,43.111111,85.545959,1460.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
1,2376792.0,9.0,3.0,752.0,0.0,188.0,0.0,83.555556,99.084700,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7
2,131350.0,10.0,3.0,7564.0,0.0,1460.0,0.0,756.400000,690.497277,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
3,164796.0,6.0,3.0,770.0,0.0,385.0,0.0,128.333333,198.813145,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3
4,163418.0,6.0,3.0,400.0,0.0,200.0,0.0,66.666667,103.279556,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6


In [ ]:
logging.info("Data Preprocessing...")

In [ ]:
labels = {
  "Benign": 0,
  "Analysis": 1,
  "Backdoor": 2,
  "DoS": 3,
  "Exploits": 4,
  "Fuzzers": 5,
  "Generic": 6,
  "Reconnaissance": 7,
  "Shellcode": 8,
  "Worms": 9
}

In [ ]:
test_attacks = ['Analysis', 'Backdoor', 'Shellcode', 'Worms']

In [ ]:
df_benign = df[df['Label'] == labels['Benign']]

In [ ]:
test_labels = [labels[attack] for attack in test_attacks if attack in labels]
df_test_attacks = df[df['Label'].isin(test_labels)]

In [ ]:
df_train_attacks = df[(df['Label'] != labels['Benign']) & (~df['Label'].isin(test_labels))]

In [ ]:
b_train, b_temp = train_test_split(df_benign, test_size=0.30, random_state=RAND)

In [ ]:
b_val, b_test = train_test_split(b_temp, test_size=0.50, random_state=RAND)

In [ ]:
a_train, a_val = train_test_split(df_train_attacks, test_size=0.20, random_state=RAND)

In [ ]:
df_train = pd.concat([b_train, a_train]).sample(frac=1, random_state=RAND).reset_index(drop=True)
df_val = pd.concat([b_val, a_val]).sample(frac=1, random_state=RAND).reset_index(drop=True)

In [ ]:
df_test = pd.concat([b_test, df_test_attacks]).sample(frac=1, random_state=RAND).reset_index(drop=True)

In [ ]:
df_train['Label'] = df_train['Label'].apply(lambda x: min(x, 1))
df_val['Label']   = df_val['Label'].apply(lambda x: min(x, 1))
df_test['Label']  = df_test['Label'].apply(lambda x: min(x, 1))

In [ ]:
print("--- Tamanhos ---")
print(f"Treino: {len(df_train)} | Validação: {len(df_val)} | Teste: {len(df_test)}")
print("\n--- Ataques no Teste ---")
print(df_test['Label'].value_counts())

--- Tamanhos ---
Treino: 319950 | Validação: 71030 | Teste: 56935

--- Ataques no Teste ---
Label
0    53750
8     2102
2      452
1      385
9      246
Name: count, dtype: int64


In [ ]:
df_train['Label'].unique(), df_val['Label'].unique(), df_test['Label'].unique()

(array([6, 0, 5, 4, 7, 3]), array([0, 5, 4, 7, 6, 3]), array([0, 8, 2, 9, 1]))

In [ ]:
X_train = df_train.drop('Label', axis=1)
y_train = df_train['Label']
X_val = df_val.drop('Label', axis=1)
y_val = df_val['Label']
X_test = df_test.drop('Label', axis=1)
y_test = df_test['Label']

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((319950, 76), (319950,), (56935, 76), (56935,))

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),
         make_column_selector(dtype_include=['number']))
    ])

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)


In [ ]:
with open('./models/preprocessor_v1.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

In [ ]:

logging.info(f"X_train shape: {X_train.shape}")

In [ ]:
BATCH_SIZE=1024
EPOCHS=1000

In [ ]:
logging.info("Starting hyperparameter optimization")

In [ ]:
strategy = tf.distribute.MirroredStrategy()

def objective(trial):
  with strategy.scope():
    inputs = tf.keras.Input(shape=(X_train.shape[1],))

    x = Dense(1024, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(768, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)

    # Camada de atenção
    attention = Dense(512, activation='softmax')(x)
    x = Multiply()([x, attention])

    output = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=lr),metrics=['accuracy'])

  early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,
      restore_best_weights=True
  )


  model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=0
  )


  loss, accuracy = model.evaluate(X_val, y_val, verbose=0)

  return accuracy

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-05-27 16:44:18,118] A new study created in memory with name: no-name-de4abf46-6308-4df8-9956-6b688370c201


NameError: name 'objective' is not defined

In [ ]:
logging.info("Best hyperparameters:")
for key, value in study.best_params.items():
  logging.info(f"{key}: {value}")

In [ ]:
best_lr = study.best_params['lr']

In [ ]:
checkpoint = ModelCheckpoint(
    './checkpoints/checkpoint.keras',
    monitor='val_loss',
    save_best_only=True
)


In [ ]:
logging.info("Starting model training")

In [ ]:
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
  inputs = tf.keras.Input(shape=(X_train.shape[1],))

  x = Dense(1024, activation='relu')(inputs)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(768, activation='relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(512, activation='relu')(x)
  x = Dropout(0.2)(x)

  # Camada de atenção
  attention = Dense(512, activation='softmax')(x)
  x = Multiply()([x, attention])

  output = Dense(1, activation='sigmoid')(x)

  model = tf.keras.Model(inputs=inputs, outputs=output)

  model.summary()

  model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=best_lr),metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 1024)           │        78,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 768)            │       787,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           513 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │             2 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,260,291 (4.81 MB)

 Trainable params: 1,260,291 (4.81 MB)

 Non-trainable params: 0 (0.00 B)

NameError: name 'best_lr' is not defined

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
  X_train, y_train,
  batch_size=BATCH_SIZE,
  epochs=EPOCHS,
  validation_data=(X_val, y_val),
  callbacks=[early_stop, checkpoint],
  verbose=1
)
model.save('./models/model_new_cic_unsw-nb15-v1.keras')

Epoch 1/1000
 469/5040 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - accuracy: 0.9575 - loss: 0.1504

KeyboardInterrupt: 

In [ ]:
logging.info("Done. Evaluating")

In [ ]:

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred , average="binary")
precision = precision_score(y_test, y_pred , average="binary")
f1 = f1_score(y_test, y_pred, average="binary")
print("----------------------------------------------")
print("accuracy")
print("%.3f" %accuracy)
print("recall")
print("%.3f" %recall)
print("precision")
print("%.3f" %precision)
print("f1score")
print("%.3f" %f1)

In [ ]:
results = {
    "accuracy": float(accuracy),
    "recall": float(recall),
    "precision": float(precision),
    "f1": float(f1)
}
with open('./models/results_v1.json', 'w') as f:
    json.dump(results, f, indent=2)